# 03 — Modelling and Validation

This notebook trains and compares models that predict ciprofloxacin resistance from the genomic
features extracted in notebook 02.

| Research question | Addressed in this notebook by |
|---|---|
| **RQ3** Which model performs best? | Majority-class and rule-based baselines vs Logistic Regression, Random Forest and XGBoost, compared with AUROC, balanced accuracy and MCC |
| **RQ2** Do models generalise to genetically distinct strains? | Random cross-validation vs clade-aware cross-validation (whole lineages held out), plus performance on held-out isolates from lineages never seen in training |
| **RQ1** Which features matter? | Ablation: all features vs known resistance mechanisms only (SHAP analysis follows in notebook 04) |

**Preventing data leakage**
- The original train/test split from Pataki et al. is kept. The 266 test isolates are not used for any
  modelling decision and are evaluated once, at the end.
- Rare-feature filtering and hyperparameter tuning happen inside each training fold (nested CV).
- Lineage clusters are computed from genome distances only, never from the resistance labels.

## 1. Settings

In [ ]:
from pathlib import Path

DATA_DIR = Path("Data")
RESULTS_DIR = Path("results")
MODELS_DIR = Path("models")

# Resistance label. EUCAST clinical breakpoint for ciprofloxacin in Enterobacterales:
# resistant if MIC > 0.5 mg/L. (CLSI alternative: resistant if MIC >= 1 mg/L)
BREAKPOINT = 0.5

# Isolate quality control (from notebook 02)
MIN_COVERAGE = 10      # minimum estimated genome coverage
MIN_BREADTH = 0.95     # minimum fraction of gyrA and parC covered by >= 5 reads

# Lineage clusters: Mash distance at which the genome-distance tree is cut
# (0.01 is roughly 99% average nucleotide identity)
CLADE_DISTANCE = 0.01
CLADE_DISTANCES_TO_TEST = [0.005, 0.01, 0.02]

N_FOLDS = 5
MIN_FEATURE_COUNT = 3  # features present in fewer isolates are dropped inside each fold
RANDOM_SEED = 42
N_JOBS = -1

In [ ]:
import time
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

from src.amr_utils import (CATEGORY_LABELS, METRICS, KnownMechanismRule, MinCountFilter,
                           bootstrap_ci, evaluate, feature_category,
                           known_mechanism_columns, lineage_clusters)

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 30)
pd.set_option("display.precision", 3)
sns.set_theme(style="whitegrid")

FIG_DIR = RESULTS_DIR / "figures"
for d in [RESULTS_DIR, FIG_DIR, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print("Libraries imported successfully.")

## 2. Load the data
- `supplementary_meta_table.csv`: MIC values and the original train/test split
- `features.csv`, `qc.csv`, `mash_distances.csv`: outputs of notebook 02

In [ ]:
metadata = pd.read_csv(DATA_DIR / "supplementary_meta_table.csv")
features = pd.read_csv(DATA_DIR / "features.csv")
qc = pd.read_csv(DATA_DIR / "qc.csv")
distances = pd.read_csv(DATA_DIR / "mash_distances.csv", index_col=0)

print("Isolates with MIC values:", len(metadata))
print("Isolates with extracted features:", len(features))
print("Genomic features:", features.shape[1] - 1)

# Quality control: remove isolates with low coverage of the key target genes
qc["passed_qc"] = ((qc["estimated_coverage"] >= MIN_COVERAGE) &
                   (qc["gyrA_breadth"] >= MIN_BREADTH) &
                   (qc["parC_breadth"] >= MIN_BREADTH))
print("Isolates failing QC:", (~qc["passed_qc"]).sum())

data = (metadata[["sample_alias", "country", "measurement", "train_test"]]
        .merge(qc.loc[qc["passed_qc"], ["sample_alias"]], on="sample_alias")
        .merge(features, on="sample_alias")
        .set_index("sample_alias"))
data["resistant"] = (data["measurement"] > BREAKPOINT).astype(int)

# Lineage clustering needs a genome distance for every isolate
no_distance = ~data.index.isin(distances.index)
print("Isolates without a Mash sketch (excluded):", no_distance.sum())
data = data[~no_distance]

feature_cols = [c for c in features.columns if c != "sample_alias"]
print("\nIsolates used for modelling:", len(data))
display(data.groupby("train_test")["resistant"].agg(isolates="count", resistant="sum", proportion_resistant="mean"))

## 3. Resistance labels
Ciprofloxacin MICs are typically bimodal: a susceptible population at low MICs and a resistant
population at high MICs. Few isolates close to the breakpoint means the binary label is robust.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
mic_counts = data.groupby(["measurement", "resistant"]).size().unstack(fill_value=0)
mic_counts.plot(kind="bar", stacked=True, ax=axes[0], color=["#4C72B0", "#C44E52"], width=0.85)
axes[0].set_xlabel("Ciprofloxacin MIC (mg/L)")
axes[0].set_ylabel("Isolates")
axes[0].set_title(f"MIC distribution (resistant: MIC > {BREAKPOINT} mg/L)")
axes[0].legend(["Susceptible", "Resistant"])

split_counts = data.groupby(["train_test", "resistant"]).size().unstack(fill_value=0)
split_counts.plot(kind="bar", ax=axes[1], color=["#4C72B0", "#C44E52"], rot=0)
axes[1].set_xlabel("Original split")
axes[1].set_title("Class balance per split")
axes[1].legend(["Susceptible", "Resistant"])
plt.tight_layout()
plt.savefig(FIG_DIR / "mic_distribution.png", dpi=200)
plt.show()

near = data["measurement"].between(BREAKPOINT / 2, BREAKPOINT * 2)
print(f"Isolates within one doubling dilution of the breakpoint: {near.sum()} ({near.mean():.1%})")

## 4. Feature overview
Each feature is labelled as a known resistance mechanism (QRDR target-site mutation, acquired gene,
efflux regulator truncation) or as another substitution, most of which are lineage-associated
polymorphisms. This labelling is used in the ablation below and in the SHAP analysis.

In [ ]:
feature_info = pd.DataFrame({
    "feature": feature_cols,
    "category": [feature_category(f) for f in feature_cols],
    "isolates_with_feature": data[feature_cols].sum().to_numpy(),
})
feature_info["% in resistant"] = 100 * data.loc[data.resistant == 1, feature_cols].mean().to_numpy()
feature_info["% in susceptible"] = 100 * data.loc[data.resistant == 0, feature_cols].mean().to_numpy()
feature_info.to_csv(RESULTS_DIR / "feature_overview.csv", index=False)

print("Features per category:")
display(feature_info["category"].map(CATEGORY_LABELS).value_counts().to_frame("features"))
print("Most common known resistance mechanisms:")
display(feature_info[feature_info.category != "other"]
        .sort_values("isolates_with_feature", ascending=False).head(15).round(1))

## 5. Lineage clusters
Isolates are grouped into lineages by hierarchical clustering of their Mash genome distances.
Clade-aware cross-validation keeps every isolate of a lineage in the same fold, so the model is always
tested on lineages it has not seen. If resistance is concentrated in a few lineages, a model can score
well in random CV by recognising the lineage instead of the resistance mechanism.

In [ ]:
distances = distances.loc[data.index, data.index]
data["clade"] = lineage_clusters(distances, CLADE_DISTANCE)

clade_summary = (data.groupby("clade")
                 .agg(isolates=("resistant", "size"), proportion_resistant=("resistant", "mean"))
                 .sort_values("isolates", ascending=False))
print(f"Lineage clusters at Mash distance {CLADE_DISTANCE}: {len(clade_summary)}")
print(f"Singleton clusters: {(clade_summary.isolates == 1).sum()}")
print(f"Largest cluster: {clade_summary.isolates.max()} isolates "
      f"({clade_summary.isolates.max() / len(data):.1%})")

top = clade_summary.head(15)
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(top.index.astype(str), top["isolates"], color="#BBBBBB", label="Isolates")
ax2 = ax.twinx()
ax2.plot(top.index.astype(str), top["proportion_resistant"], "o", color="#C44E52", label="Proportion resistant")
ax2.set_ylim(0, 1.05)
ax.set_xlabel("Lineage cluster (15 largest)")
ax.set_ylabel("Isolates")
ax2.set_ylabel("Proportion resistant")
ax.set_title("Resistance by lineage")
fig.legend(loc="upper right", bbox_to_anchor=(0.9, 0.88))
plt.tight_layout()
plt.savefig(FIG_DIR / "resistance_by_lineage.png", dpi=200)
plt.show()

## 6. Models
| Model | Role |
|---|---|
| Majority class | Lower bound: always predicts the most common class |
| Known-mechanism rule | Literature baseline: resistant if an isolate carries at least *k* known mechanisms (k chosen on the training data) |
| Logistic Regression | Interpretable linear model (L1 or L2 regularisation) |
| Random Forest | Ensemble of decision trees, captures interactions between mutations |
| XGBoost | Gradient-boosted trees |

Each ML model is a pipeline (rare-feature filter → model) tuned with an inner 3-fold grid search on
AUROC, so tuning only ever sees the training fold. Class weights compensate for class imbalance.

In [ ]:
def tuned(model, grid):
    pipeline = Pipeline([("filter", MinCountFilter(MIN_FEATURE_COUNT)), ("model", model)])
    inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)
    return GridSearchCV(pipeline, {f"model__{k}": v for k, v in grid.items()},
                        cv=inner_cv, scoring="roc_auc", n_jobs=N_JOBS)

def make_models():
    return {
        "Majority class": DummyClassifier(strategy="most_frequent"),
        "Known-mechanism rule": KnownMechanismRule(),
        "Logistic Regression": tuned(
            LogisticRegression(solver="liblinear", class_weight="balanced", max_iter=5000),
            {"C": [0.01, 0.1, 1, 10], "penalty": ["l1", "l2"]}),
        "Random Forest": tuned(
            RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                   random_state=RANDOM_SEED, n_jobs=1),
            {"max_depth": [None, 6], "min_samples_leaf": [1, 3], "max_features": ["sqrt", 0.3]}),
        "XGBoost": tuned(
            XGBClassifier(n_estimators=300, eval_metric="logloss", random_state=RANDOM_SEED, n_jobs=1),
            {"max_depth": [2, 4], "learning_rate": [0.05, 0.1], "subsample": [0.8, 1.0]}),
    }

ML_MODELS = ["Logistic Regression", "Random Forest", "XGBoost"]
MODEL_ORDER = list(make_models())

## 7. Random vs clade-aware cross-validation
Both schemes use 5 folds on the development set (original training isolates) with stratification by
class. In clade-aware CV, all isolates from one lineage cluster are placed in the same fold.

In [ ]:
dev = data[data.train_test == "train"]
test = data[data.train_test == "test"]
X_dev, y_dev, groups_dev = dev[feature_cols], dev["resistant"], dev["clade"]
X_test, y_test = test[feature_cols], test["resistant"]
print(f"Development set: {len(dev)} isolates ({y_dev.mean():.1%} resistant), {groups_dev.nunique()} lineages")
print(f"Held-out test set: {len(test)} isolates ({y_test.mean():.1%} resistant)")

SPLITTERS = {
    "Random CV": StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED),
    "Clade-aware CV": StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED),
}

def cross_validate(models, X, y, groups, scheme):
    rows = []
    for name, estimator in models.items():
        start = time.time()
        for fold, (train_idx, test_idx) in enumerate(SPLITTERS[scheme].split(X, y, groups)):
            model = clone(estimator).fit(X.iloc[train_idx], y.iloc[train_idx])
            prob = model.predict_proba(X.iloc[test_idx])[:, 1]
            rows.append({"scheme": scheme, "model": name, "fold": fold,
                         "test_isolates": len(test_idx), **evaluate(y.iloc[test_idx], prob)})
        print(f"  {scheme:15s} {name:22s} {time.time() - start:6.1f}s")
    return rows

cv_rows = []
for scheme in SPLITTERS:
    cv_rows += cross_validate(make_models(), X_dev, y_dev, groups_dev, scheme)
cv_results = pd.DataFrame(cv_rows)
cv_results.to_csv(RESULTS_DIR / "cv_fold_results.csv", index=False)

In [ ]:
def summarise(results, by):
    summary = results.groupby(by)[METRICS].agg(["mean", "std"])
    formatted = pd.DataFrame(index=summary.index)
    for m in METRICS:
        formatted[m] = (summary[(m, "mean")].map("{:.3f}".format) + " ± " +
                        summary[(m, "std")].map("{:.3f}".format))
    return formatted

cv_summary = summarise(cv_results, ["scheme", "model"])
cv_summary = cv_summary.reindex(pd.MultiIndex.from_product([SPLITTERS, MODEL_ORDER]))
cv_summary.to_csv(RESULTS_DIR / "cv_summary.csv")
print("Mean ± standard deviation across the 5 folds")
display(cv_summary)

In [ ]:
plot_data = cv_results.melt(id_vars=["scheme", "model"], value_vars=["AUROC", "balanced_accuracy", "MCC"],
                            var_name="metric", value_name="score")
g = sns.catplot(data=plot_data, x="model", y="score", hue="scheme", col="metric", kind="bar",
                order=MODEL_ORDER, errorbar="sd", height=4, aspect=1.15,
                palette=["#4C72B0", "#DD8452"])
g.set_xticklabels(rotation=35, ha="right")
g.set_axis_labels("", "Score")
g.figure.suptitle("Random vs clade-aware cross-validation (mean ± SD over 5 folds)", y=1.03)
plt.savefig(FIG_DIR / "cv_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

means = cv_results.groupby(["model", "scheme"])[["AUROC", "balanced_accuracy", "MCC"]].mean().unstack()
gap = pd.DataFrame({m: means[(m, "Random CV")] - means[(m, "Clade-aware CV")]
                    for m in ["AUROC", "balanced_accuracy", "MCC"]}).loc[MODEL_ORDER]
gap.to_csv(RESULTS_DIR / "generalisation_gap.csv")
print("Generalisation gap (random CV minus clade-aware CV; positive = worse on unseen lineages)")
display(gap.round(3))

## 8. Ablation: known resistance mechanisms only
If the models mainly rely on established mechanisms, removing all other substitutions (likely lineage
markers) should barely change clade-aware performance. A large drop would show that lineage markers
carry predictive information beyond the known mechanisms.

In [ ]:
known_cols = known_mechanism_columns(feature_cols)
print(f"Known-mechanism features: {len(known_cols)} of {len(feature_cols)}")

ml_models = {k: v for k, v in make_models().items() if k in ML_MODELS}
ablation_rows = []
for scheme in SPLITTERS:
    ablation_rows += [dict(r, features="Known mechanisms only")
                      for r in cross_validate(ml_models, X_dev[known_cols], y_dev, groups_dev, scheme)]
ablation = pd.concat([
    cv_results[cv_results.model.isin(ML_MODELS)].assign(features="All features"),
    pd.DataFrame(ablation_rows),
])
ablation.to_csv(RESULTS_DIR / "ablation_fold_results.csv", index=False)
ablation_summary = summarise(ablation, ["scheme", "model", "features"])[["AUROC", "balanced_accuracy", "MCC"]]
ablation_summary.to_csv(RESULTS_DIR / "ablation_summary.csv")
display(ablation_summary)

## 9. Sensitivity to the lineage definition
The clade-aware result depends on how coarsely lineages are defined. XGBoost is re-evaluated with
lineage clusters cut at several genome-distance thresholds (larger distance = fewer, broader lineages
= a stricter test).

In [ ]:
sensitivity_rows = []
for threshold in CLADE_DISTANCES_TO_TEST:
    groups = lineage_clusters(distances, threshold).loc[dev.index]
    rows = cross_validate({"XGBoost": make_models()["XGBoost"]}, X_dev, y_dev, groups, "Clade-aware CV")
    for r in rows:
        r.update(clade_distance=threshold, lineages=groups.nunique())
    sensitivity_rows += rows
sensitivity = (pd.DataFrame(sensitivity_rows)
               .groupby(["clade_distance", "lineages"])[["AUROC", "balanced_accuracy", "MCC"]].agg(["mean", "std"]))
sensitivity.to_csv(RESULTS_DIR / "clade_threshold_sensitivity.csv")
display(sensitivity.round(3))

## 10. Final models and held-out test set
Each model is tuned and trained on the full development set, then evaluated once on the 266 held-out
isolates. 95% confidence intervals come from 1,000 bootstrap resamples of the test set.

The test isolates come from different studies than the development isolates, so this is also a test of
transfer to a new population. Results are additionally split by whether an isolate's lineage was present
in the development set.

In [ ]:
final_models, test_probs = {}, {}
for name, estimator in make_models().items():
    model = clone(estimator).fit(X_dev, y_dev)
    final_models[name] = model.best_estimator_ if isinstance(model, GridSearchCV) else model
    test_probs[name] = final_models[name].predict_proba(X_test)[:, 1]
    if isinstance(model, GridSearchCV):
        params = {k.replace("model__", ""): v for k, v in model.best_params_.items()}
        print(f"{name}: best parameters {params}")
    joblib.dump(final_models[name], MODELS_DIR / f"{name.lower().replace(' ', '_').replace('-', '_')}.joblib")

print(f"Known-mechanism rule threshold: >= {final_models['Known-mechanism rule'].threshold_} mechanisms")

holdout = pd.concat({name: bootstrap_ci(y_test, prob, seed=RANDOM_SEED)
                     for name, prob in test_probs.items()}, names=["model", "metric"])
holdout.to_csv(RESULTS_DIR / "holdout_results.csv")
holdout_table = holdout.apply(lambda r: f"{r.estimate:.3f} ({r.ci_low:.3f}–{r.ci_high:.3f})", axis=1).unstack()
display(holdout_table[METRICS].loc[MODEL_ORDER])

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
for name in MODEL_ORDER[1:]:
    RocCurveDisplay.from_predictions(y_test, test_probs[name], name=name, ax=ax)
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_title("ROC curves on the held-out test set")
plt.tight_layout()
plt.savefig(FIG_DIR / "roc_holdout.png", dpi=200)
plt.show()

In [ ]:
seen_lineages = set(dev["clade"])
test_seen = test["clade"].isin(seen_lineages)
print(f"Test isolates from lineages seen in development: {test_seen.sum()}; unseen: {(~test_seen).sum()}")

rows = []
for name in MODEL_ORDER:
    for label, mask in [("Seen lineage", test_seen), ("Unseen lineage", ~test_seen)]:
        if mask.sum() >= 10:
            rows.append({"model": name, "subset": label, "isolates": int(mask.sum()),
                         **evaluate(y_test[mask], test_probs[name][mask.to_numpy()])})
lineage_holdout = pd.DataFrame(rows).set_index(["model", "subset"])
lineage_holdout.to_csv(RESULTS_DIR / "holdout_by_lineage.csv")
display(lineage_holdout.round(3))

## 11. Save everything needed by notebook 04 (SHAP)

In [ ]:
joblib.dump({
    "X_dev": X_dev, "y_dev": y_dev, "X_test": X_test, "y_test": y_test,
    "clades": data["clade"], "metadata": data[["country", "measurement", "train_test"]],
    "feature_cols": feature_cols, "breakpoint": BREAKPOINT, "clade_distance": CLADE_DISTANCE,
}, MODELS_DIR / "datasets.joblib")
print("Saved models:", sorted(p.name for p in MODELS_DIR.glob("*.joblib")))
print("Saved results:", sorted(p.name for p in RESULTS_DIR.glob("*.csv")))